In [1]:
# %% --- Cell 1: Install Dependencies ---
!pip install torch torch-geometric scikit-learn -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.2 MB/s eta 0:00:00


In [2]:
# %% --- Cell 2: Upload Dataset Files ---
import json, os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score, classification_report
from torch_geometric.data import HeteroData
from torch_geometric.nn import GCNConv
from torch_geometric.utils import negative_sampling

# --- Upload files from local machine (option 1) ---
from google.colab import files
uploaded = files.upload()  # upload all 8 JSON files

# --- Or mount Google Drive (option 2) ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/Thesis'  # <-- adjust path

# --- For local testing ---
DATA_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else '/content'


Saving a_writes.json to a_writes.json
Saving advisors.json to advisors.json
Saving belongs_to.json to belongs_to.json
Saving courses.json to courses.json
Saving experts_in.json to experts_in.json
Saving interested_in.json to interested_in.json
Saving papers.json to papers.json
Saving research_area.json to research_area.json
Saving students.json to students.json
Saving takes.json to takes.json


In [3]:
# %% --- Cell 3: Data Loading ---
def load_json(name):
    with open(os.path.join(DATA_DIR, name), 'r', encoding='utf-8') as f:
        return json.load(f)

advisors_raw = load_json('advisors.json')
students_raw = load_json('students.json')
papers_raw = load_json('papers.json')
courses_raw = load_json('courses.json')
research_areas_raw = load_json('research_area.json')

# Deduplicate entities to prevent mapping mismatch out-of-bounds errors
seen_adv = set()
advisors = []
for a in advisors_raw:
    if a.get('name') and a['name'] not in seen_adv:
        seen_adv.add(a['name'])
        advisors.append(a)

seen_stu = set()
students = []
for s in students_raw:
    if s.get('student_id') and s['student_id'] not in seen_stu:
        seen_stu.add(s['student_id'])
        students.append(s)

seen_pap = set()
papers = []
for p in papers_raw:
    idx = p.get('paper_index')
    if idx is not None and idx not in seen_pap:
        seen_pap.add(idx)
        papers.append(p)

seen_crs = set()
courses = []
for c in courses_raw:
    cid = c.get('course_id')
    if cid is not None and cid not in seen_crs:
        seen_crs.add(cid)
        courses.append(c)

seen_ra = set()
research_areas = []
for r in research_areas_raw:
    rid = r.get('id')
    if rid is not None and rid not in seen_ra:
        seen_ra.add(rid)
        research_areas.append(r)

a_writes = load_json('a_writes.json')
belongs_to = load_json('belongs_to.json')
experts_in = load_json('experts_in.json')
interested_in = load_json('interested_in.json')
takes = load_json('takes.json')

print(f"Advisors: {len(advisors)}, Students: {len(students)}, Papers: {len(papers)}")
print(f"Courses: {len(courses)}, Research Areas: {len(research_areas)}")


Advisors: 630, Students: 482, Papers: 9384
Courses: 538, Research Areas: 44


In [4]:
# %% --- Cell 4: Node Index Mappings ---
adv_map = {a['name']: i for i, a in enumerate(advisors)}
stu_map = {s['student_id']: i for i, s in enumerate(students)}
pap_map = {p['paper_index']: i for i, p in enumerate(papers)}
crs_map = {c['course_id']: i for i, c in enumerate(courses)}
ra_map  = {r['id']: i for i, r in enumerate(research_areas)}


In [5]:
# %% --- Cell 5: Feature Engineering ---
# Global vocabulary for multi-hot encoding
vocab = set()
for a in advisors:
    vocab.update(a.get('publication_topics', []))
    vocab.update(a.get('research_areas', []))
for s in students:
    vocab.update(s.get('research_interests', []))
for r in research_areas:
    vocab.add(r['research_area'])
vocab.discard('')
vocab = sorted(vocab)
vocab_map = {t: i for i, t in enumerate(vocab)}
V = len(vocab)

def multi_hot(terms, dim=V):
    vec = np.zeros(dim, dtype=np.float32)
    for t in terms:
        if t in vocab_map: vec[vocab_map[t]] = 1.0
    return vec

# --- Advisor features ---
desigs = sorted(set(a['designation'] for a in advisors))
desig_map = {d: i for i, d in enumerate(desigs)}
D = len(desigs)

adv_feats = []
for a in advisors:
    dv = np.zeros(D, dtype=np.float32); dv[desig_map[a['designation']]] = 1.0
    pc = float(a.get('publication_count', 0) or 0)
    cap = float(a.get('capacity', 0) or 0)
    mh = multi_hot(a.get('publication_topics', []) + a.get('research_areas', []))
    adv_feats.append(np.concatenate([dv, [pc, cap], mh]))
adv_feats = np.stack(adv_feats)
adv_feats[:, D:D+2] = MinMaxScaler().fit_transform(adv_feats[:, D:D+2])
adv_x = torch.tensor(adv_feats, dtype=torch.float)

# --- Student features ---
stu_feats = []
for s in students:
    cgpa = float(s.get('cgpa', 0) or 0) / 4.0
    cv = np.zeros(len(crs_map), dtype=np.float32)
    for c in s.get('completed_courses', []):
        if c in crs_map: cv[crs_map[c]] = 1.0
    iv = multi_hot(s.get('research_interests', []))
    stu_feats.append(np.concatenate([[cgpa], cv, iv]))
stu_x = torch.tensor(np.stack(stu_feats), dtype=torch.float)

# --- Paper features (TF-IDF) ---
titles = [p['paper_title'] for p in papers]
tfidf_p = TfidfVectorizer(max_features=128, stop_words='english', sublinear_tf=True)
pap_x = torch.tensor(tfidf_p.fit_transform(titles).toarray(), dtype=torch.float)

# --- Course features ---
tfidf_c = TfidfVectorizer(max_features=32, stop_words='english')
crs_text = tfidf_c.fit_transform([c['course_name'] for c in courses]).toarray().astype(np.float32)
crs_x = torch.tensor(np.concatenate([np.eye(len(courses), dtype=np.float32), crs_text], axis=1))

# --- Research Area features ---
ra_feats = []
for i, r in enumerate(research_areas):
    eye = np.zeros(len(research_areas), dtype=np.float32); eye[i] = 1.0
    ra_feats.append(np.concatenate([eye, multi_hot([r['research_area']])]))
ra_x = torch.tensor(np.stack(ra_feats), dtype=torch.float)

print("Feature shapes:")
for name, feat in [('advisor', adv_x), ('student', stu_x), ('paper', pap_x), ('course', crs_x), ('research_area', ra_x)]:
    print(f"  {name:15s} {list(feat.shape)}")


Feature shapes:
  advisor         [630, 5282]
  student         [482, 5813]
  paper           [9384, 128]
  course          [538, 570]
  research_area   [44, 5318]


In [6]:
# %% --- Cell 6: Build Edge Indices ---
def build_edges(records, src_key, dst_key, src_map, dst_map):
    s, d = [], []
    for e in records:
        sk, dk = e.get(src_key), e.get(dst_key)
        if sk in src_map and dk in dst_map:
            s.append(src_map[sk]); d.append(dst_map[dk])
    return torch.tensor([s, d], dtype=torch.long) if s else torch.zeros((2, 0), dtype=torch.long)

# Name-to-RA-id lookup
ra_name2id = {r['research_area']: r['id'] for r in research_areas}

writes_ei = build_edges(a_writes, 'advisor', 'paper_index', adv_map, pap_map)
belongs_ei = build_edges(belongs_to, 'paper_index', 'research_area_id', pap_map, ra_map)

# experts_in: advisor name → research_area name → ra_id → ra_map index
exp_src, exp_dst = [], []
for e in experts_in:
    a, area = e.get('advisor'), e.get('research_area')
    if isinstance(area, list): continue
    aid = ra_name2id.get(area)
    if a in adv_map and aid in ra_map:
        exp_src.append(adv_map[a]); exp_dst.append(ra_map[aid])
expert_ei = torch.tensor([exp_src, exp_dst], dtype=torch.long)

int_src, int_dst = [], []
for e in interested_in:
    sid, area = e.get('student_id'), e.get('research_area')
    aid = ra_name2id.get(area)
    if sid in stu_map and aid is not None and aid in ra_map:
        int_src.append(stu_map[sid]); int_dst.append(ra_map[aid])
interest_ei = torch.tensor([int_src, int_dst], dtype=torch.long)

takes_ei = build_edges(takes, 'student_id', 'course_id', stu_map, crs_map)

print("\nEdge counts:")
for name, ei in [('writes', writes_ei), ('belongs_to', belongs_ei), ('expert_in', expert_ei),
                  ('interested_in', interest_ei), ('takes', takes_ei)]:
    print(f"  {name:15s} {ei.size(1)}")



Edge counts:
  writes          9284
  belongs_to      11438
  expert_in       778
  interested_in   1419
  takes           2196


In [7]:
# %% --- Cell 7: Build HeteroData ---
hdata = HeteroData()
hdata['advisor'].x = adv_x;          hdata['advisor'].num_nodes = adv_x.size(0)
hdata['student'].x = stu_x;          hdata['student'].num_nodes = stu_x.size(0)
hdata['paper'].x = pap_x;            hdata['paper'].num_nodes = pap_x.size(0)
hdata['course'].x = crs_x;           hdata['course'].num_nodes = crs_x.size(0)
hdata['research_area'].x = ra_x;     hdata['research_area'].num_nodes = ra_x.size(0)

edge_defs = [
    ('advisor', 'writes', 'paper', writes_ei),
    ('paper', 'belongs_to', 'research_area', belongs_ei),
    ('advisor', 'expert_in', 'research_area', expert_ei),
    ('student', 'interested_in', 'research_area', interest_ei),
    ('student', 'takes', 'course', takes_ei),
]
for s, r, d, ei in edge_defs:
    hdata[s, r, d].edge_index = ei
    hdata[d, f'rev_{r}', s].edge_index = ei.flip(0)

print("\nHeteroData:", hdata)



HeteroData: HeteroData(
  advisor={
    x=[630, 5282],
    num_nodes=630,
  },
  student={
    x=[482, 5813],
    num_nodes=482,
  },
  paper={
    x=[9384, 128],
    num_nodes=9384,
  },
  course={
    x=[538, 570],
    num_nodes=538,
  },
  research_area={
    x=[44, 5318],
    num_nodes=44,
  },
  (advisor, writes, paper)={ edge_index=[2, 9284] },
  (paper, rev_writes, advisor)={ edge_index=[2, 9284] },
  (paper, belongs_to, research_area)={ edge_index=[2, 11438] },
  (research_area, rev_belongs_to, paper)={ edge_index=[2, 11438] },
  (advisor, expert_in, research_area)={ edge_index=[2, 778] },
  (research_area, rev_expert_in, advisor)={ edge_index=[2, 778] },
  (student, interested_in, research_area)={ edge_index=[2, 1419] },
  (research_area, rev_interested_in, student)={ edge_index=[2, 1419] },
  (student, takes, course)={ edge_index=[2, 2196] },
  (course, rev_takes, student)={ edge_index=[2, 2196] }
)


In [8]:
# %% --- Cell 8: Homogeneous Projection ---
TARGET_DIM = 128
node_types = list(hdata.node_types)
projected, type_labels, node_offsets = [], [], {}
offset = 0

for i, nt in enumerate(node_types):
    feat = hdata[nt].x
    n, d = feat.shape
    proj = nn.Linear(d, TARGET_DIM, bias=False)
    nn.init.xavier_uniform_(proj.weight)
    with torch.no_grad():
        projected.append(proj(feat))
    type_labels.extend([i] * n)
    node_offsets[nt] = offset
    offset += n

homo_x = torch.cat(projected, dim=0)
type_labels = torch.tensor(type_labels, dtype=torch.long)

all_src, all_dst = [], []
for (s, r, d) in hdata.edge_types:
    ei = hdata[s, r, d].edge_index
    all_src.append(ei[0] + node_offsets[s])
    all_dst.append(ei[1] + node_offsets[d])
homo_edge_index = torch.stack([torch.cat(all_src), torch.cat(all_dst)])

print(f"\nHomogeneous graph: {homo_x.size(0)} nodes, {homo_edge_index.size(1)} edges, {TARGET_DIM}d features")
print(f"Node type mapping: {dict(zip(node_types, range(len(node_types))))}")




Homogeneous graph: 11078 nodes, 50230 edges, 128d features
Node type mapping: {'advisor': 0, 'student': 1, 'paper': 2, 'course': 3, 'research_area': 4}


In [9]:
# %% --- Cell 9: GCN Models ---
class GCNEncoder(nn.Module):
    def __init__(self, in_ch, hid_ch, out_ch, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hid_ch)
        self.conv2 = GCNConv(hid_ch, out_ch)
        self.bn = nn.BatchNorm1d(hid_ch)
        self.drop = dropout
    def forward(self, x, ei):
        x = F.dropout(F.relu(self.bn(self.conv1(x, ei))), p=self.drop, training=self.training)
        return self.conv2(x, ei)

class LinkPredictor(nn.Module):
    def __init__(self, dim, hid=64):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(2*dim, hid), nn.ReLU(), nn.Dropout(0.3), nn.Linear(hid, 1))
    def forward(self, zs, zd):
        return self.mlp(torch.cat([zs, zd], -1)).squeeze(-1)

class GCNLinkPred(nn.Module):
    def __init__(self, in_ch, hid=128, emb=64):
        super().__init__()
        self.enc = GCNEncoder(in_ch, hid, emb)
        self.pred = LinkPredictor(emb)
    def encode(self, x, ei): return self.enc(x, ei)
    def decode(self, z, s, d): return self.pred(z[s], z[d])

class GCNNodeClf(nn.Module):
    def __init__(self, in_ch, hid, n_cls):
        super().__init__()
        self.enc = GCNEncoder(in_ch, hid, hid)
        self.clf = nn.Linear(hid, n_cls)
    def forward(self, x, ei): return self.clf(self.enc(x, ei))


In [10]:
# %% --- Cell 10: Train Link Prediction ---
def split_edges(ei, n_nodes, test_r=0.15, val_r=0.05):
    perm = torch.randperm(ei.size(1))
    nt, nv = int(ei.size(1)*test_r), int(ei.size(1)*val_r)
    return (ei[:, perm[nt+nv:]], ei[:, perm[nt:nt+nv]],
            negative_sampling(ei, n_nodes, num_neg_samples=nv),
            ei[:, perm[:nt]],
            negative_sampling(ei, n_nodes, num_neg_samples=nt))

print("\n" + "="*60)
print(" GCN LINK PREDICTION")
print("="*60)

train_ei, vp, vn, tp, tn = split_edges(homo_edge_index, homo_x.size(0))
print(f"Train: {train_ei.size(1)}, Val: {vp.size(1)}, Test: {tp.size(1)}")

lp_model = GCNLinkPred(TARGET_DIM)
opt = torch.optim.Adam(lp_model.parameters(), lr=0.01, weight_decay=5e-4)
crit = nn.BCEWithLogitsLoss()
best_auc, best_st = 0, None

for ep in range(1, 201):
    lp_model.train(); opt.zero_grad()
    z = lp_model.encode(homo_x, train_ei)
    ps = lp_model.pred(z[train_ei[0]], z[train_ei[1]])
    ne = negative_sampling(train_ei, homo_x.size(0), num_neg_samples=train_ei.size(1))
    ns = lp_model.pred(z[ne[0]], z[ne[1]])
    loss = crit(torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)]))
    loss.backward(); opt.step()

    if ep % 20 == 0 or ep == 1:
        lp_model.eval()
        with torch.no_grad():
            z = lp_model.encode(homo_x, train_ei)
            vps = lp_model.pred(z[vp[0]], z[vp[1]])
            vns = lp_model.pred(z[vn[0]], z[vn[1]])
            sc = torch.cat([vps, vns]).cpu().numpy()
            lb = np.concatenate([np.ones(vps.size(0)), np.zeros(vns.size(0))])
            auc = roc_auc_score(lb, 1/(1+np.exp(-sc)))
            ap = average_precision_score(lb, 1/(1+np.exp(-sc)))
        if auc > best_auc: best_auc = auc; best_st = lp_model.state_dict().copy()
        print(f"  Ep {ep:>4d}  Loss {loss.item():.4f}  Val AUC {auc:.4f}  AP {ap:.4f}")

# Test
lp_model.load_state_dict(best_st); lp_model.eval()
with torch.no_grad():
    z = lp_model.encode(homo_x, train_ei)
    tps = lp_model.pred(z[tp[0]], z[tp[1]])
    tns = lp_model.pred(z[tn[0]], z[tn[1]])
    sc = torch.cat([tps, tns]).cpu().numpy()
    lb = np.concatenate([np.ones(tps.size(0)), np.zeros(tns.size(0))])
    test_auc = roc_auc_score(lb, 1/(1+np.exp(-sc)))
    test_ap = average_precision_score(lb, 1/(1+np.exp(-sc)))
print(f"\n  TEST AUC: {test_auc:.4f}  |  TEST AP: {test_ap:.4f}")

# Save embeddings
with torch.no_grad():
    final_z = lp_model.encode(homo_x, homo_edge_index)
print(f"  Node embeddings shape: {list(final_z.shape)}")



 GCN LINK PREDICTION
Train: 40185, Val: 2511, Test: 7534
  Ep    1  Loss 0.7598  Val AUC 0.9583  AP 0.9609
  Ep   20  Loss 0.2498  Val AUC 0.9568  AP 0.9600
  Ep   40  Loss 0.1887  Val AUC 0.9759  AP 0.9745
  Ep   60  Loss 0.1469  Val AUC 0.9832  AP 0.9813
  Ep   80  Loss 0.1358  Val AUC 0.9859  AP 0.9839
  Ep  100  Loss 0.1192  Val AUC 0.9870  AP 0.9849
  Ep  120  Loss 0.1151  Val AUC 0.9878  AP 0.9861
  Ep  140  Loss 0.1111  Val AUC 0.9883  AP 0.9866
  Ep  160  Loss 0.1041  Val AUC 0.9879  AP 0.9862
  Ep  180  Loss 0.1106  Val AUC 0.9868  AP 0.9853
  Ep  200  Loss 0.1020  Val AUC 0.9887  AP 0.9872

  TEST AUC: 0.9895  |  TEST AP: 0.9879
  Node embeddings shape: [11078, 64]


In [11]:
# %% --- Cell 11: Train Node Classification ---
print("\n" + "="*60)
print(" GCN NODE CLASSIFICATION")
print("="*60)

N = homo_x.size(0); n_cls = type_labels.max().item() + 1
perm = torch.randperm(N)
nt2, nv2 = int(0.15*N), int(0.05*N)
test_m = torch.zeros(N, dtype=torch.bool); test_m[perm[:nt2]] = True
val_m = torch.zeros(N, dtype=torch.bool);  val_m[perm[nt2:nt2+nv2]] = True
train_m = torch.zeros(N, dtype=torch.bool); train_m[perm[nt2+nv2:]] = True

nc_model = GCNNodeClf(TARGET_DIM, 128, n_cls)
opt2 = torch.optim.Adam(nc_model.parameters(), lr=0.01, weight_decay=5e-4)
best_va, best_st2 = 0, None

for ep in range(1, 201):
    nc_model.train(); opt2.zero_grad()
    out = nc_model(homo_x, homo_edge_index)
    loss = F.cross_entropy(out[train_m], type_labels[train_m])
    loss.backward(); opt2.step()

    if ep % 20 == 0 or ep == 1:
        nc_model.eval()
        with torch.no_grad():
            pred = nc_model(homo_x, homo_edge_index).argmax(1)
            ta = accuracy_score(type_labels[train_m].numpy(), pred[train_m].numpy())
            va = accuracy_score(type_labels[val_m].numpy(), pred[val_m].numpy())
        if va > best_va: best_va = va; best_st2 = nc_model.state_dict().copy()
        print(f"  Ep {ep:>4d}  Loss {loss.item():.4f}  Train {ta:.4f}  Val {va:.4f}")

nc_model.load_state_dict(best_st2); nc_model.eval()
with torch.no_grad():
    pred = nc_model(homo_x, homo_edge_index).argmax(1)
    ta2 = accuracy_score(type_labels[test_m].numpy(), pred[test_m].numpy())
    tf2 = f1_score(type_labels[test_m].numpy(), pred[test_m].numpy(), average='macro')
print(f"\n  TEST Accuracy: {ta2:.4f}  |  TEST F1: {tf2:.4f}")
print(classification_report(type_labels[test_m].numpy(), pred[test_m].numpy(),
      target_names=node_types, zero_division=0))



 GCN NODE CLASSIFICATION
  Ep    1  Loss 1.8915  Train 0.8469  Val 0.8535
  Ep   20  Loss 0.0908  Train 0.9439  Val 0.9566
  Ep   40  Loss 0.0567  Train 0.9673  Val 0.9675
  Ep   60  Loss 0.0415  Train 0.9897  Val 0.9873
  Ep   80  Loss 0.0358  Train 0.9935  Val 0.9819
  Ep  100  Loss 0.0299  Train 0.9946  Val 0.9765
  Ep  120  Loss 0.0279  Train 0.9968  Val 0.9765
  Ep  140  Loss 0.0244  Train 0.9973  Val 0.9801
  Ep  160  Loss 0.0230  Train 0.9970  Val 0.9801
  Ep  180  Loss 0.0215  Train 0.9974  Val 0.9801
  Ep  200  Loss 0.0202  Train 0.9976  Val 0.9819

  TEST Accuracy: 0.9795  |  TEST F1: 0.8132
               precision    recall  f1-score   support

      advisor       0.86      0.83      0.84        98
      student       1.00      1.00      1.00        75
        paper       1.00      0.99      0.99      1405
       course       0.96      0.97      0.97        75
research_area       0.20      0.38      0.26         8

     accuracy                           0.98      1661
   

In [12]:
# %% --- Cell 12: Save All ---
os.makedirs('processed', exist_ok=True)
torch.save(hdata, 'processed/academic_graph_hetero.pt')
torch.save({'x': homo_x, 'edge_index': homo_edge_index,
            'node_type_labels': type_labels, 'node_offsets': node_offsets},
           'processed/academic_graph_homo.pt')
torch.save(final_z, 'processed/node_embeddings.pt')
torch.save(best_st, 'processed/gcn_link_pred_model.pt')
torch.save(best_st2, 'processed/gcn_node_clf_model.pt')
print("\nAll saved to processed/ directory!")
print("  - academic_graph_hetero.pt")
print("  - academic_graph_homo.pt")
print("  - node_embeddings.pt")
print("  - gcn_link_pred_model.pt")
print("  - gcn_node_clf_model.pt")



All saved to processed/ directory!
  - academic_graph_hetero.pt
  - academic_graph_homo.pt
  - node_embeddings.pt
  - gcn_link_pred_model.pt
  - gcn_node_clf_model.pt
